# 01 — Pseudo-Mask Generation

Generates the pseudo-label dataset used to fine-tune the cross-lead attention Stage 2 model.

Pipeline: raw ECG image → Stage 0 normalization → Stage 1 rectification → pretrained shared-conv2d LeadModel → sparse COO pseudo-mask.


In [ ]:
# Kaggle-only setup. Requires the competition data and public baseline checkpoints.
!pip install --no-deps segmentation-models-pytorch==0.5.0
!pip install connected-components-3d --no-index --find-links=file:///kaggle/input/hengck23-demo-submit-physionet/setup/


In [ ]:
import os, sys, cv2, torch
import numpy as np
import pandas as pd
from timeit import default_timer as timer

sys.path.insert(0, '/kaggle/input/my-stage2-lead-model')
sys.path.append('/kaggle/input/hengck23-demo-submit-physionet')
sys.path.append('/kaggle/input/physionet-final-submission-models')

from stage0_common import time_to_str
from stage0_model import Net as Stage0Net
from stage1_model import Net as Stage1Net
from stage2_lead_model import Net as LeadModel
from stage2_common import *
import stage0_common as s0c
import stage1_common as s1c


In [ ]:
MODE = 'local'
DEVICE = 'cuda'
FLOAT_TYPE = torch.float16
KAGGLE_DIR = '/kaggle/input/physionet-ecg-image-digitization'
WEIGHT_DIR = '/kaggle/input/hengck23-demo-submit-physionet/weight'
OUT_DIR = '/kaggle/working/output'
WINDOW_SIZE = 240
OFFSET = 416
xscale = 5000 / (2080 - 118)
IMGH, IMGW = int(1700), int(2200 * xscale + 1)
x0, x1 = 0, 5600
y0, y1 = 0, 1696
zero_mv = [703.5, 987.5, 1271.5, 1531.5]
os.makedirs(f'{OUT_DIR}/rectified', exist_ok=True)
os.makedirs(f'{OUT_DIR}/masks', exist_ok=True)


In [ ]:
def save_sparse_mask_coo(mask_dense, save_path, threshold=0.3):
    arrays = {'shape': np.array(mask_dense.shape)}
    for i in range(mask_dense.shape[0]):
        ys, xs = np.where(mask_dense[i] > threshold)
        arrays[f'ch{i}_y'] = ys.astype(np.int32)
        arrays[f'ch{i}_x'] = xs.astype(np.int32)
        arrays[f'ch{i}_v'] = mask_dense[i, ys, xs].astype(np.float32)
    np.savez_compressed(save_path, **arrays)

def read_lead_crops(rectified_path):
    image = cv2.imread(rectified_path, cv2.IMREAD_COLOR)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (IMGW, IMGH), interpolation=cv2.INTER_LINEAR)
    image = image[y0:y1, x0:x1]
    H, W, _ = image.shape
    crops = []
    for zmv in zero_mv:
        h0, h1 = int(zmv - WINDOW_SIZE), int(zmv + WINDOW_SIZE)
        src_h0, src_h1 = max(0, h0), min(H, h1)
        dst_h0 = src_h0 - h0
        dst_h1 = dst_h0 + (src_h1 - src_h0)
        crop = np.zeros((WINDOW_SIZE * 2, W, 3), np.uint8)
        crop[dst_h0:dst_h1] = image[src_h0:src_h1]
        crops.append(crop)
    return np.stack(crops)


The full project run saved rectified JPEG images and sparse `.mask-coo.npz` masks. Final log: 8,792 usable samples and one Stage 1 failure: `3728731089-0012`.
